# COF-Landscaper Example: ILCOF-1

This notebook demonstrates the standard COF-Landscaper workflow using ILCOF-1, from framework construction and interlayer stacking analysis to geometry optimisation and comparison of simulated and experimental powder X-ray diffraction (PXRD) patterns.

The workflow uses the default calculation settings and focuses on the chemical understanding and interpretation of the results. For a more technical introduction, the [configurable notebook](../configurable/configurable.ipynb) provides detailed explanations of alternative calculators, optimisation settings, stacking parameters, candidate selection and additional analysis tools.

For installation and setup instructions, refer to the [README](../../../../README.md). Open this notebook from its own directory and select the Python kernel associated with your COF-Landscaper environment.

In [ ]:
import coflandscaper as cl

## Define the framework

In this example, we model ILCOF-1 as a two-dimensional square (`sql`) network consisting of four-connected nodes and two-connected linkers.

**Choosing the building fragments is part of the flexibility of COF-Landscaper.** The input structures do not need to correspond to the molecular precursors used in the experimental synthesis. Instead, you can divide the final framework into convenient fragments, provided their connection sites allow them to reconstruct the intended chemical connectivity within the chosen topology. The manual construction route therefore does not require a predefined reaction for each linkage chemistry.

The illustration below shows **one possible choice of fragments for ILCOF-1**, using the input files supplied with this notebook. The teal node contains the pyrene core and its four surrounding phenyl rings. The purple linker contains the central phenyl ring and both imine groups. In this partition, the cuts lie at the single bonds between the outer phenyl rings and the imine carbon atoms. The imine bonds themselves are already present in the prepared linker.

![One possible division of ILCOF-1 into a four-connected node and a two-connected linker, with helium markers identifying the connection sites](figures/ilcof1_building_fragments.svg)

Helium (`He`) atoms mark the connection sites in each input fragment: four on the node and two on the linker. These atoms serve only as placeholders to indicate where neighbouring fragments should connect. They are removed during assembly and are not present in the resulting COF structure.

The prepared fragments used here are provided in:

- **Node:** `0_node/ILCOF-1_node.xyz`
- **Linker:** `0_linker/ILCOF-1_linker.xyz`

Other divisions of the same framework are possible. What matters is that the fragments have suitable connection-site geometry and reproduce the intended framework connectivity when joined.

[Original ILCOF-1 synthesis](https://doi.org/10.1002/chem.201203753)

In [ ]:
COF_NAME = "ILCOF-1"
TOPOLOGY = "sql"

## Construct a single layer

We first generate an idealised single-layer structure of ILCOF-1. This provides the starting geometry for the subsequent optimisation of atomic positions and lattice parameters.

The resulting structure is saved as `ILCOF-1_unopt.cif` in the single-layer folder.

In [ ]:
cl.BuildCOF2D().build(TOPOLOGY, COF_NAME)

## Optimise the single layer

We next optimise the atomic positions and unit-cell parameters to obtain a relaxed single-layer structure for the subsequent stacking analysis.

The optimised structure is saved as `ILCOF-1_preopt.cif` in the single-layer folder.

In [ ]:
cl.GeometryOpt().run_preopt(COF_NAME)

## Explore how the layers stack

The relative arrangement of neighbouring COF layers can strongly influence their stability, pore geometry and powder X-ray diffraction (PXRD) patterns. Rather than considering only perfectly eclipsed (AA) or staggered (AB) stacking, COF-Landscaper systematically explores intermediate arrangements to identify energetically favourable structures.

Each stacking arrangement is described by two parameters:

- **Interlayer distance (ILD):** The vertical separation between neighbouring layers.
- **Interlayer slip (ILS):** The lateral displacement of one layer relative to another.

![Schematic illustration of interlayer distance, interlayer slip and the resulting energy landscape](figures/stacking_concept.svg)

For ILCOF-1, we sample interlayer distances from **3.0 to 4.0 Å** in 0.1 Å steps and lateral displacements from AA towards AB stacking along the `(a+b)/2` direction, using 1 Å steps and including the exact AB position.

Both inclined and serrated stacking arrangements are considered. The resulting structures are saved in `ILCOF-1/2_ILCOF-1_matrix/` and form the basis of the energy landscape calculated in the next step.

In [ ]:
cl.CreateMatrix().run(COF_NAME, topo=TOPOLOGY)

## Calculate single-point energies

We calculate the energy of each stacking arrangement while keeping its atomic positions and unit cell unchanged. Comparing these energies reveals which combinations of layer separation and lateral displacement are energetically favourable, forming the basis of the energy landscape.

Separate energy tables for inclined and serrated stacking are saved in `ILCOF-1/3_ILCOF-1_landscape/`.

In [ ]:
cl.SinglePoint().run(COF_NAME)

## View the landscapes

The landscapes show how the relative energy changes with interlayer distance and slip. Each star marks the lowest-energy sampled arrangement for that stacking mode.

These maps provide an initial picture of the stacking preferences of the prepared layer. They explore a selected part of the potential-energy surface; they do not establish the fully relaxed global minimum or cover every possible framework conformation. The results depend on the layer geometry, including the orientations and twists of its nodes and linkers, as well as the chosen sampling range and direction.

**Control over the starting structure is a useful part of this approach.** By adjusting the prepared fragments, you can explore chemically motivated alternatives and investigate how different starting conformations influence the stacking preferences.

The aim is to **identify promising starting geometries efficiently before carrying out more detailed optimisations**. Sampling the landscape helps locate favourable arrangements beyond the conventional AA and AB starting points.

If several low-energy regions, or **basins**, are visible, structures from different regions can be selected for separate optimisation. This allows us to investigate whether they lead to distinct relaxed structures or converge towards the same minimum.

The landscape figures are saved in `ILCOF-1/3_ILCOF-1_landscape/`.

In [ ]:
cl.Landscape().run(COF_NAME, show=True)

## Select candidates

This step determines which stacking arrangements will undergo geometry optimisation. By default, we select the **global minimum of each sampled landscape**, corresponding to the lowest-energy evaluated structure in each stacking mode.

You can also select additional candidates, such as structures from other low-energy regions, to investigate how they behave during relaxation.

The selected CIF files are saved in `ILCOF-1/3_ILCOF-1_landscape/selection/`.

In [ ]:
cl.SelectCofs().run(COF_NAME)

## Relax the stacked candidates

We optimise the atomic positions and all unit-cell parameters of each selected structure. This allows the framework geometry, layer separation and lateral displacement to adjust together, providing relaxed candidates for the subsequent structural analysis and comparison with experiment.

The optimised CIF files and energies per layer are saved in `ILCOF-1/4_ILCOF-1_optimization/`.

In [ ]:
cl.GeometryOpt().run(COF_NAME)

## Inspect the optimised structures

Inclined and serrated stacking represent idealised starting arrangements used to explore the stacking landscape. During geometry optimisation, these structures can evolve towards different arrangements or develop characteristics of both stacking modes.

Explore the optimised structures interactively by rotating and zooming in on the framework to inspect the relative arrangement of neighbouring layers.

Although this approach allows us to explore stacking arrangements beyond conventional AA and AB models, the resulting structures remain idealised periodic models. Real COFs may exhibit more complex stacking sequences, structural disorder and local variations in layer displacement that cannot be fully captured within a small periodic unit cell.

In [ ]:
cl.AnalyzeStacking().analyze(COF_NAME)

In [ ]:
cl.VisualizeCOF().visualize_cof(COF_NAME)

## Compare diffraction patterns

We simulate the powder X-ray diffraction (PXRD) patterns of the optimised structures and compare them with the experimental pattern to assess how well the proposed stacking arrangements describe the measured data.

**Choosing the correct wavelength is essential**, as the diffraction peak positions depend on the wavelength of the incident X-rays. Check the instrument settings or experimental data before selecting the corresponding wavelength for the simulation.

COF-Landscaper uses the radiation definitions provided by pymatgen. Common choices include:

| Radiation | Wavelength (Å) | Input |
|---|---:|---|
| Cu Kα₁ | 1.54056 | `CuKa1` |
| Cu Kα | 1.54184 | `CuKa` |
| Cu Kα₂ | 1.54439 | `CuKa2` |
| Mo Kα₁ | 0.70930 | `MoKa1` |
| Co Kα₁ | 1.78896 | `CoKa1` |

A complete list of supported radiation sources is available in the [pymatgen diffraction documentation](https://pymatgen.org/pymatgen.analysis.diffraction.html#pymatgen.analysis.diffraction.xrd.XRDCalculator).

For ILCOF-1, the supplied experimental data were collected using a wavelength of 1.5406 Å. We therefore select `CuKa1`.

The simulated diffraction patterns are saved as XY files in `ILCOF-1/5_ILCOF-1_analysis/pxrd_xy/`. The corresponding `_hkl.csv` files provide the Miller indices (hkl) and diffraction angles of the ten strongest calculated reflections, allowing individual peaks to be associated with crystallographic plane families.

When interpreting the comparison, remember that the simulated patterns represent idealised periodic structures. Differences from experiment may arise from stacking disorder, structural defects, guest molecules or other features not captured by the structural model. Good agreement supports a proposed stacking arrangement but does not necessarily establish a unique structural assignment.

In [ ]:
cl.PXRD(wavelength="CuKa1").run(COF_NAME, export_hkl=True)

## Visualise the diffraction patterns

Compare the simulated PXRD patterns of the optimised structures with the measured pattern provided in `experimental_pxrd/`.

To inspect individual reflections or zoom in on a particular region, you can specify the displayed 2θ range using the `xlim` parameter. For example, `xlim=(2, 10)` displays only the region between 2° and 10°.

The comparison figures are saved in `ILCOF-1/5_ILCOF-1_analysis/pxrd_plots/`. Each new plot replaces the previous figure for the same structure. If you wish to retain comparisons over different angular ranges, rename the saved PDFs before generating new ones.

Changing the displayed range does not affect the underlying simulated diffraction data.

In [ ]:
cl.PXRD().plot_sim_vs_exp(COF_NAME, xlim=(2, 30))